# Feature Engineering with information available at renewal

## Overview

- The goal of this notebook is to engineer features from variables of dataset that have been obtained by an underwriter(s) at least first renewal. This implies that we may now have more internal data available our our disposal.
- This will be tackled in 4 waves in terms of the data now available to us
    - Driver Features
    - Payment Features
    - Claims Features
    - Vehicle Features

## Setup

In [2]:
import os
from datetime import datetime
from typing import Any
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
os.chdir('..') ## on the assumption that this is rum from notebook/freq_sev

### Load the data

In [3]:
insurance_initiation_variables_path = "data/input/Motor_vehicle_insurance_data.csv"
claims_variables_path = "data/input/exp/sample_type_claim.csv"
insurance_df = pd.read_csv(insurance_initiation_variables_path, delimiter=';')
claims_df = pd.read_csv(claims_variables_path, delimiter=';')

## 1. Prepare Dataset

| Step | Action |
|------|--------|
| Aggregate | Sum claims by (ID, year) |
| Merge | Left join on (ID, year) |
| Fill | NaN → 0 for no claims |
| Split | 80/20 train-test split |

### 1.1 Aggregate claims by policyholder and year

In [5]:
claim_grouping_columns = ['ID', 'Cost_claims_year']
claim_aggregation_column = 'Cost_claims_by_type'
claims_aggregated = (
    claims_df
    .groupby(claim_grouping_columns, as_index=False)[claim_aggregation_column]
    .sum()
)

### 1.2 Merge insurance and claims data

In [6]:
merging_columns = ['ID', 'Cost_claims_year']

dataset = insurance_df.merge(claims_aggregated, on=merging_columns, how='left')
dataset[claim_aggregation_column] = dataset[claim_aggregation_column].fillna(0)
dataset['claims_frequency'] = (dataset[claim_aggregation_column] > 0).astype(int)

### 1.3 Split into train and test sets

In [ ]:
test_ratio = 0.2
to_shuffle = False
if to_shuffle:
    dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

split_index = int(len(dataset) * (1 - test_ratio))
trainset = dataset.iloc[:split_index].reset_index(drop=True)
testset = dataset.iloc[split_index:].reset_index(drop=True)
print(f"Total records: {len(dataset)}")
print(f"Training set: {len(trainset)} records ({100*(1-test_ratio):.0f}%)")
print(f"Test set: {len(testset)} records ({100*test_ratio:.0f}%)")

## 2. Feature Engineering

While there are now advancement in the motor insurance sector that enables using telematic data points, I will be exploring traditional features in this segment mostly based on domain knowledge [some capture here](https://www.researchgate.net/publication/338007809_An_Analysis_of_the_Risk_Factors_Determining_Motor_Insurance_Premium_in_a_Small_Island_State_The_Case_of_Malta).


### 2.1 - Driver Features
The below are directly available in the dataset
- Driver DOB
- Driver Driving License date
- Second driver on the car

Some features to engineer from the above
- Respective driver age at inception of each contract
- Evidence backed driver age banding at the inception of each contract
- Length of driving license (driving experience) at the inception of each contract
- Evidence backed license length banding at the inception of each contract.
- Interaction between second driver & driver age banding - as a proxyto identify couples or parent/children?
    - e.g young single driver may be considered high risk
    - older driver with second driver may be low to moderate risk (although we cant conclude with this)
- Age experience gap for each contract. Difference between age and driving experience in years. Lower is better? But how low?
    - So a 25 year old with 1 year experience has agegapexp = 24
    - 25 years old with 6 years experience has agegapexp=19

In [7]:
dataset['Date_last_renewal'] = pd.to_datetime(dataset['Date_last_renewal'], format='%d/%m/%Y', dayfirst=True)
dataset['Date_birth'] = pd.to_datetime(dataset['Date_birth'], format='%d/%m/%Y', dayfirst=True)
dataset['driver_age_at_contract_start'] =  dataset['Date_last_renewal'].dt.year - dataset['Date_birth'].dt.year
dataset['Date_driving_licence'] = pd.to_datetime(dataset['Date_driving_licence'], format='%d/%m/%Y', yearfirst=True)
dataset['driver_experience_age'] = dataset['Date_last_renewal'].dt.year - dataset['Date_driving_licence'].dt.year
dataset['driver_age_experience_age_diff'] = abs(dataset['driver_age_at_contract_start'] - dataset['driver_experience_age'])
dataset['driver_age_experience_ratio_proxy_for_driving_experience'] = dataset['driver_age_experience_age_diff'] / dataset['driver_age_at_contract_start']
dataset[['claims_frequency', 'driver_age_experience_ratio_proxy_for_driving_experience']].corr(method='pearson')


,claims_frequency,driver_age_experience_ratio_proxy_for_driving_experience
claims_frequency,1.000000,0.049575
driver_age_experience_ratio_proxy_for_driving_experience,0.049575,1.000000


Having tried all the above features, the engineered features showing the best relationship is driving age experience ratio.
- Calculated by - (difference between drivers age at contract inception and total driving experience in years) / drivers age at contract inception
- This gives a ratio between 0 and 1. Values closer to 1 mean that driver is inexperienced while values closer to zero indicates very experienced drivers
- So for example
    - Driver A with driver age 19years and driving experience 1year. Ratio = (19-1)/19 = 0.95 (inexperienced driver)
    - Driver B with driver age 30years and driving experience 12 years. Ratio = (30-12)/30 = 0.6 (decently experienced driver)

So this feature simply says how experienced is the driver relative to their opportunity to have gained experience

### Payment Features
The below are directly available in the dataset as payment related attributes
- Lapse (number of policy cancelled due to default for a single maturing year)
- Date Lapse (default date)
- Payment (annually=0 or half-year administrative process=1)

Premium excluded for obvious leakage reasons.

Some feature engineering ideas
- Bin the lapse into 0 and 1+ lapse. This will capture customers that have not cancelled for any reason or have had to terminate a contract within the year. A signal for dissatisfaction or financial distress?
- Use payment as it is.



In [ ]:
dataset['Payment'].value_counts(normalize=True) * 100

In [ ]:
(
    dataset[['Payment', 'Cost_claims_year','ID']]
    .groupby('Payment').agg(
    {
        'ID':'count',
        'Cost_claims_year':'mean',
    }
    )
    .rename(columns={'ID':'policy_count'})

)

In [ ]:
dataset['lapse_flag'] = (dataset['Lapse'] >= 1).astype(int)
dataset[['lapse_flag', 'claims_frequency']].corr(method='pearson')

- Payment would be making it as a solid feature candidate. Despite half-year payment mode being used for circa 30% of the dataset, the average claims cost for this group is 2x that of the annual payment group.
- Lapse flag - still in the back pocket